# 07 – Smoothing × K × Classifier: Results

**Hypothesis:** Explicit Gaussian smoothing has measurable effect on classification
only when K is large enough for the basis to encode high-frequency content.

**Grid:** K ∈ {10, 50}, basis ∈ {chebyshev, legendre, bspline},
σ ∈ {0, 0.5, 1, 2, 3, 5, 10, 20}, classifiers = {LR, XGBoost},
50-fold RSKF (10 rep × 5 folds).

In [40]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats

from _common import RESULTS_DIR

raw = pd.read_csv(RESULTS_DIR / "smoothing_kxclf_raw.csv")

METRIC_COLS = [
    "roc_auc", "pr_auc", "f1", "sensitivity", "precision",
    "specificity", "accuracy", "youden_j", "brier", "log_loss",
]
GROUP_COLS = ["K", "basis", "sigma", "classifier"]
BASIS_COLORS = {"bspline": "#1f77b4", "chebyshev": "#ff7f0e", "legendre": "#2ca02c"}
BASES = ["bspline", "chebyshev", "legendre"]

agg = {}
for col in METRIC_COLS:
    agg[col] = ["mean", "std"]
summary = raw.groupby(GROUP_COLS, sort=True).agg(agg)
summary.columns = [f"{col}_{stat}" for col, stat in summary.columns]
summary = summary.reset_index()

K_LIST = sorted(raw["K"].unique())
CLF_LIST = sorted(raw["classifier"].unique())
SIGMAS_NONZERO = [s for s in sorted(raw["sigma"].unique()) if s > 0]


def plot_metric_vs_sigma(metric: str, title: str):
    """Line plot of metric vs sigma, faceted by K (rows) x classifier (cols)."""
    mean_col = f"{metric}_mean"
    std_col = f"{metric}_std"
    fig = make_subplots(
        rows=len(K_LIST), cols=len(CLF_LIST),
        subplot_titles=[f"K={k}, {c}" for k in K_LIST for c in CLF_LIST],
        shared_xaxes=True, shared_yaxes=True,
        vertical_spacing=0.10, horizontal_spacing=0.06,
    )
    for ri, K in enumerate(K_LIST, 1):
        for ci, clf in enumerate(CLF_LIST, 1):
            for basis in BASES:
                sub = summary[
                    (summary["K"] == K) & (summary["classifier"] == clf) & (summary["basis"] == basis)
                ].sort_values("sigma")
                fig.add_trace(
                    go.Scatter(
                        x=sub["sigma"], y=sub[mean_col],
                        error_y=dict(type="data", array=sub[std_col].values, visible=True, thickness=1),
                        mode="lines+markers", name=basis,
                        line=dict(color=BASIS_COLORS[basis]),
                        legendgroup=basis,
                        showlegend=(ri == 1 and ci == 1),
                    ),
                    row=ri, col=ci,
                )
    fig.update_xaxes(title_text="Gaussian σ (pixels)", row=len(K_LIST))
    fig.update_yaxes(title_text=f"{title} (mean ± std)", col=1)
    fig.update_layout(height=600, width=900, title_text=f"{title} vs Gaussian smoothing σ", template="plotly_white")
    return fig


def plot_delta_heatmap(metric: str, title: str):
    """Heatmap of Δ metric relative to σ=0 baseline, faceted by K x classifier."""
    mean_col = f"{metric}_mean"
    fig = make_subplots(
        rows=len(K_LIST), cols=len(CLF_LIST),
        subplot_titles=[f"K={k}, {c}" for k in K_LIST for c in CLF_LIST],
        vertical_spacing=0.12, horizontal_spacing=0.08,
    )
    all_deltas = []
    for ri, K in enumerate(K_LIST, 1):
        for ci, clf in enumerate(CLF_LIST, 1):
            delta_matrix = []
            for basis in BASES:
                baseline = summary[
                    (summary["K"] == K) & (summary["classifier"] == clf)
                    & (summary["basis"] == basis) & (summary["sigma"] == 0)
                ][mean_col].values[0]
                row_deltas = []
                for sigma in SIGMAS_NONZERO:
                    val = summary[
                        (summary["K"] == K) & (summary["classifier"] == clf)
                        & (summary["basis"] == basis) & (summary["sigma"] == sigma)
                    ][mean_col].values[0]
                    row_deltas.append(val - baseline)
                delta_matrix.append(row_deltas)
            all_deltas.extend([d for row in delta_matrix for d in row])
            fig.add_trace(
                go.Heatmap(
                    z=delta_matrix, x=[str(s) for s in SIGMAS_NONZERO], y=BASES,
                    colorscale="RdYlGn", zmid=0,
                    text=[[f"{d:.4f}" for d in row] for row in delta_matrix],
                    texttemplate="%{text}",
                    showscale=(ri == 1 and ci == len(CLF_LIST)),
                    colorbar=dict(title=f"Δ {title}") if (ri == 1 and ci == len(CLF_LIST)) else None,
                ),
                row=ri, col=ci,
            )
    abs_max = max(abs(d) for d in all_deltas) if all_deltas else 0.01
    fig.update_traces(zmin=-abs_max, zmax=abs_max)
    fig.update_xaxes(title_text="σ", row=len(K_LIST))
    fig.update_yaxes(title_text="basis", col=1)
    fig.update_layout(height=500, width=950, title_text=f"Δ {title} relative to no smoothing (σ=0)", template="plotly_white")
    return fig


print(f"Raw: {len(raw)} rows, Summary: {len(summary)} rows")
print(f"K values: {K_LIST}")
print(f"Classifiers: {CLF_LIST}")
print(f"Bases: {BASES}")
print(f"Sigmas: {sorted(raw['sigma'].unique())}")
print(f"Splits: {raw['split'].nunique()}")

Raw: 4800 rows, Summary: 96 rows
K values: [np.int64(10), np.int64(50)]
Classifiers: ['LR', 'XGB']
Bases: ['bspline', 'chebyshev', 'legendre']
Sigmas: [np.float64(0.0), np.float64(0.5), np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(5.0), np.float64(10.0), np.float64(20.0)]
Splits: 50


## 1. Overview: Mean PR-AUC by K × Classifier × Sigma

Averaged across all 3 bases and 50 splits.

In [41]:
overview = (
    raw.groupby(["K", "classifier", "sigma"])["pr_auc"]
    .agg(["mean", "std"])
    .reset_index()
)

pivot = overview.pivot_table(index="sigma", columns=["K", "classifier"], values="mean")
pivot.columns = [f"K={k} {c}" for k, c in pivot.columns]
pivot = pivot.round(4)
pivot

,K=10 LR,K=10 XGB,K=50 LR,K=50 XGB
sigma,,,,
0.0,0.8776,0.8519,0.8725,0.8263
0.5,0.8777,0.8525,0.8726,0.8309
1.0,0.8776,0.8517,0.8727,0.8384
2.0,0.8781,0.8527,0.8734,0.8514
3.0,0.8780,0.8547,0.8741,0.8575
5.0,0.8780,0.8564,0.8763,0.8621
10.0,0.8791,0.8548,0.8762,0.8628
20.0,0.8786,0.8562,0.8758,0.8574


## 2. PR-AUC vs Sigma — by K and Classifier

One subplot per (K, classifier) combination. Each line is a basis type.
If smoothing matters, lines should show a clear trend with sigma.

In [42]:
plot_metric_vs_sigma("pr_auc", "PR-AUC").show()

## 3. ROC-AUC vs Sigma

In [43]:
plot_metric_vs_sigma("roc_auc", "ROC-AUC").show()

## 3b. Sensitivity vs Sigma

In [44]:
plot_metric_vs_sigma("sensitivity", "Sensitivity").show()

## 3c. Precision vs Sigma

In [45]:
plot_metric_vs_sigma("precision", "Precision").show()

## 4. Δ PR-AUC Heatmaps: Effect of Smoothing Relative to Baseline (σ=0)

Each heatmap shows `mean PR-AUC(σ) − mean PR-AUC(σ=0)` for a given (K, classifier).
Green = smoothing helps, red = smoothing hurts.

In [46]:
plot_delta_heatmap("pr_auc", "PR-AUC").show()

## 4b. Δ Sensitivity Heatmap

In [47]:
plot_delta_heatmap("sensitivity", "Sensitivity").show()

## 4c. Δ Precision Heatmap

In [48]:
plot_delta_heatmap("precision", "Precision").show()

## 5. Statistical Significance: σ=0 vs Best σ

For each (K, basis, classifier) we identify the σ with highest mean PR-AUC,
then run a paired Wilcoxon signed-rank test comparing fold-level PR-AUC
values of σ=0 vs best-σ across all 50 splits.

This answers: *Is the best smoothing setting significantly different from no smoothing?*

In [49]:
results_stat = []

for K in K_LIST:
    for clf in CLF_LIST:
        for basis in bases:
            sub_summary = summary[
                (summary["K"] == K)
                & (summary["classifier"] == clf)
                & (summary["basis"] == basis)
            ]
            best_row = sub_summary.loc[sub_summary["pr_auc_mean"].idxmax()]
            best_sigma = best_row["sigma"]
            best_mean = best_row["pr_auc_mean"]

            baseline_mean = sub_summary[sub_summary["sigma"] == 0]["pr_auc_mean"].values[0]

            if best_sigma == 0:
                results_stat.append({
                    "K": K, "classifier": clf, "basis": basis,
                    "best_sigma": 0, "baseline_pr_auc": baseline_mean,
                    "best_pr_auc": best_mean, "delta": 0.0,
                    "p_value": 1.0, "significant": False,
                })
                continue

            pr_baseline = raw[
                (raw["K"] == K) & (raw["classifier"] == clf)
                & (raw["basis"] == basis) & (raw["sigma"] == 0)
            ].sort_values("split")["pr_auc"].values

            pr_best = raw[
                (raw["K"] == K) & (raw["classifier"] == clf)
                & (raw["basis"] == basis) & (raw["sigma"] == best_sigma)
            ].sort_values("split")["pr_auc"].values

            stat, p = stats.wilcoxon(pr_baseline, pr_best, alternative="two-sided")
            delta = best_mean - baseline_mean

            results_stat.append({
                "K": K, "classifier": clf, "basis": basis,
                "best_sigma": best_sigma, "baseline_pr_auc": round(baseline_mean, 4),
                "best_pr_auc": round(best_mean, 4), "delta": round(delta, 4),
                "p_value": round(p, 6), "significant": p < 0.05,
            })

stat_df = pd.DataFrame(results_stat)
stat_df

,K,classifier,basis,best_sigma,baseline_pr_auc,best_pr_auc,delta,p_value,significant
0,10,LR,bspline,10.0,0.8783,0.8802,0.0019,0.022378,True
1,10,LR,chebyshev,2.0,0.8791,0.8798,0.0007,0.132961,False
2,10,LR,legendre,10.0,0.8753,0.8792,0.0039,0.000030,True
3,10,XGB,bspline,20.0,0.8539,0.8654,0.0115,0.000000,True
4,10,XGB,chebyshev,5.0,0.8395,0.8488,0.0093,0.000000,True
5,10,XGB,legendre,0.5,0.8625,0.8647,0.0022,0.022975,True
6,50,LR,bspline,10.0,0.8717,0.8801,0.0084,0.000000,True
7,50,LR,chebyshev,20.0,0.8722,0.8741,0.0020,0.085793,False
8,50,LR,legendre,5.0,0.8735,0.8761,0.0027,0.043975,True
9,50,XGB,bspline,10.0,0.8522,0.8662,0.0140,0.000001,True


## 6. Effect Size: How Large Are the Differences?

Even if statistically significant, the practical effect may be negligible.
Here we show the max |Δ PR-AUC| across all sigmas for each configuration.

In [50]:
effect_rows = []
for K in K_LIST:
    for clf in CLF_LIST:
        for basis in bases:
            sub = summary[
                (summary["K"] == K)
                & (summary["classifier"] == clf)
                & (summary["basis"] == basis)
            ]
            baseline = sub[sub["sigma"] == 0]["pr_auc_mean"].values[0]
            sub_nonzero = sub[sub["sigma"] > 0]
            deltas = sub_nonzero["pr_auc_mean"].values - baseline
            effect_rows.append({
                "K": K, "classifier": clf, "basis": basis,
                "max_positive_delta": round(max(deltas), 4) if max(deltas) > 0 else 0,
                "max_negative_delta": round(min(deltas), 4) if min(deltas) < 0 else 0,
                "range": round(max(deltas) - min(deltas), 4),
            })

effect_df = pd.DataFrame(effect_rows)
print("Max absolute effect of smoothing on PR-AUC:")
effect_df

Max absolute effect of smoothing on PR-AUC:


,K,classifier,basis,max_positive_delta,max_negative_delta,range
0,10,LR,bspline,0.0019,0.0000,0.0019
1,10,LR,chebyshev,0.0007,-0.0017,0.0024
2,10,LR,legendre,0.0039,0.0000,0.0038
3,10,XGB,bspline,0.0115,-0.0031,0.0147
4,10,XGB,chebyshev,0.0093,-0.0008,0.0101
5,10,XGB,legendre,0.0022,-0.0062,0.0084
6,50,LR,bspline,0.0084,0.0000,0.0083
7,50,LR,chebyshev,0.0020,-0.0001,0.0021
8,50,LR,legendre,0.0027,-0.0014,0.0041
9,50,XGB,bspline,0.0140,-0.0006,0.0146


## 7. Best Overall Configurations (by F1)

In [51]:
top = summary.nlargest(10, "f1_mean")[
    ["K", "basis", "sigma", "classifier", "f1_mean", "f1_std",
     "sensitivity_mean", "precision_mean", "pr_auc_mean", "roc_auc_mean"]
].reset_index(drop=True)
top.columns = ["K", "basis", "σ", "clf", "F1 mean", "F1 std",
               "Sensitivity", "Precision", "PR-AUC", "ROC-AUC"]
top

,K,basis,σ,clf,F1 mean,F1 std,Sensitivity,Precision,PR-AUC,ROC-AUC
0,50,bspline,5.0,LR,0.825988,0.027020,0.818991,0.835403,0.879749,0.926490
1,10,bspline,0.0,LR,0.824886,0.027123,0.820076,0.831577,0.878294,0.923800
2,10,bspline,0.5,LR,0.824722,0.027273,0.819895,0.831434,0.878310,0.923795
3,10,bspline,1.0,LR,0.824258,0.027601,0.817930,0.832694,0.878348,0.923997
4,10,chebyshev,1.0,LR,0.824184,0.028125,0.821675,0.828381,0.879134,0.923955
5,10,bspline,20.0,LR,0.824159,0.029671,0.824191,0.825984,0.880027,0.928623
6,10,bspline,2.0,LR,0.823658,0.028642,0.819001,0.830336,0.878647,0.924281
7,10,bspline,3.0,LR,0.823318,0.029962,0.815233,0.833703,0.878437,0.924984
8,50,bspline,20.0,LR,0.822769,0.030384,0.825444,0.822187,0.879023,0.925740
9,50,bspline,10.0,LR,0.822687,0.027888,0.820595,0.826287,0.880144,0.928638


## 8. K=10 vs K=50 comparison (σ=0 baseline)

In [52]:
baseline_comparison = summary[summary["sigma"] == 0][
    ["K", "basis", "classifier", "pr_auc_mean", "pr_auc_std", "roc_auc_mean"]
].sort_values(["classifier", "basis", "K"]).reset_index(drop=True)
baseline_comparison.columns = ["K", "basis", "clf", "PR-AUC mean", "PR-AUC std", "ROC-AUC mean"]
baseline_comparison

,K,basis,clf,PR-AUC mean,PR-AUC std,ROC-AUC mean
0,10,bspline,LR,0.878294,0.024822,0.923800
1,50,bspline,LR,0.871746,0.026849,0.917156
2,10,chebyshev,LR,0.879113,0.025590,0.923893
3,50,chebyshev,LR,0.872180,0.026243,0.920719
4,10,legendre,LR,0.875334,0.024744,0.926981
5,50,legendre,LR,0.873469,0.025180,0.924635
6,10,bspline,XGB,0.853854,0.026875,0.913701
7,50,bspline,XGB,0.852159,0.027306,0.917822
8,10,chebyshev,XGB,0.839504,0.032113,0.911970
9,50,chebyshev,XGB,0.796312,0.035194,0.904970


## 9. Conclusion

Does smoothing matter? The answer depends on the magnitude and significance of the deltas above.

**Interpreting the results:**

- If Δ PR-AUC values are all < 0.005 and mostly non-significant → **smoothing is not needed**,
  the basis projection already acts as an implicit low-pass filter.
- If Δ PR-AUC is positive and significant at K=50 but not at K=10 → the hypothesis is confirmed:
  smoothing only helps when K is high enough for the basis to encode noise.
- If Δ PR-AUC is negative at large σ → aggressive smoothing destroys useful signal.

In [53]:
print("="*70)
print("  SUMMARY: Is smoothing needed?")
print("="*70)

for K in K_LIST:
    for clf in CLF_LIST:
        sub_stat = stat_df[(stat_df["K"] == K) & (stat_df["classifier"] == clf)]
        sub_eff = effect_df[(effect_df["K"] == K) & (effect_df["classifier"] == clf)]

        n_sig = sub_stat["significant"].sum()
        max_delta = sub_eff["max_positive_delta"].max()
        max_neg = sub_eff["max_negative_delta"].min()
        avg_range = sub_eff["range"].mean()

        print(f"\nK={K}, {clf}:")
        print(f"  Significant improvements over σ=0: {n_sig}/{len(sub_stat)} bases")
        print(f"  Max positive Δ PR-AUC: {max_delta:+.4f}")
        print(f"  Max negative Δ PR-AUC: {max_neg:+.4f}")
        print(f"  Avg range across σ:    {avg_range:.4f}")

        if max_delta < 0.005 and n_sig == 0:
            print("  → Smoothing has NO meaningful effect.")
        elif max_delta >= 0.005 and n_sig > 0:
            best = sub_stat.loc[sub_stat["delta"].idxmax()]
            print(f"  → Smoothing HELPS (best σ={best['best_sigma']}, "
                  f"Δ={best['delta']:+.4f}, p={best['p_value']:.4f})")
        elif max_neg < -0.005:
            print("  → Smoothing HURTS at some σ values.")
        else:
            print("  → Effect is marginal / inconclusive.")

print("\n" + "="*70)

  SUMMARY: Is smoothing needed?

K=10, LR:
  Significant improvements over σ=0: 2/3 bases
  Max positive Δ PR-AUC: +0.0039
  Max negative Δ PR-AUC: -0.0017
  Avg range across σ:    0.0027
  → Effect is marginal / inconclusive.

K=10, XGB:
  Significant improvements over σ=0: 3/3 bases
  Max positive Δ PR-AUC: +0.0115
  Max negative Δ PR-AUC: -0.0062
  Avg range across σ:    0.0111
  → Smoothing HELPS (best σ=20.0, Δ=+0.0115, p=0.0000)

K=50, LR:
  Significant improvements over σ=0: 2/3 bases
  Max positive Δ PR-AUC: +0.0084
  Max negative Δ PR-AUC: -0.0014
  Avg range across σ:    0.0048
  → Smoothing HELPS (best σ=10.0, Δ=+0.0084, p=0.0000)

K=50, XGB:
  Significant improvements over σ=0: 3/3 bases
  Max positive Δ PR-AUC: +0.0607
  Max negative Δ PR-AUC: -0.0006
  Avg range across σ:    0.0330
  → Smoothing HELPS (best σ=10.0, Δ=+0.0607, p=0.0000)



## 10. Visual: How Does the Representation Pipeline Look?

The feature pipeline for classification is: **raw spectrum → Gaussian smooth(σ) → fit basis(K coeffs) → reconstruct**.

The reconstruction from K basis coefficients is the "view" of the spectrum that the classifier actually sees.
Below we plot one star through every combination of σ and K, separated by basis.

In [54]:
import sys, importlib
sys.path.insert(0, str(RESULTS_DIR.parent))
import _common
step02 = importlib.import_module("02_generate_basis_features")

bp_block = step02.load_block(_common.BP_SAMPLED_CSV)
rp_block = step02.load_block(_common.RP_SAMPLED_CSV)

STAR_IDX = 0
ALL_SIGMAS = [0, 0.5, 1, 2, 3, 5, 10, 20]
K_VIS = [10, 50]
BASES_VIS = ["bspline", "chebyshev", "legendre"]

import plotly.express as px
SIGMA_PALETTE = px.colors.sample_colorscale(
    "Viridis", [i / (len(ALL_SIGMAS) - 1) for i in range(len(ALL_SIGMAS))]
)
SIGMA_COLORS = dict(zip(ALL_SIGMAS, SIGMA_PALETTE))

sid = int(bp_block.source_ids[STAR_IDX])
label = int(bp_block.labels[STAR_IDX])
print(f"Star: source_id={sid}, y={label}")
print(f"Pipeline: raw spectrum → gaussian_smooth(σ) → fit_basis(K) → reconstruct")
print(f"Sigmas: {ALL_SIGMAS}, K: {K_VIS}, Bases: {BASES_VIS}")

Star: source_id=1792620565667968, y=0
Pipeline: raw spectrum → gaussian_smooth(σ) → fit_basis(K) → reconstruct
Sigmas: [0, 0.5, 1, 2, 3, 5, 10, 20], K: [10, 50], Bases: ['bspline', 'chebyshev', 'legendre']


### 10a. B-spline basis

Each subplot shows the **raw spectrum** (black dashed) and the **basis reconstruction** for every σ value.
- Rows = K (number of basis coefficients)
- Columns = spectral arm (BP, RP)
- Colored lines = reconstruction from K B-spline coefficients fitted on gaussian-smoothed flux at that σ

In [55]:
def plot_basis_reconstructions(basis: str):
    fig = make_subplots(
        rows=len(K_VIS), cols=2,
        subplot_titles=[f"{arm}, K={k}" for k in K_VIS for arm in ("BP", "RP")],
        vertical_spacing=0.12, horizontal_spacing=0.06,
    )
    shown = set()

    for ri, K in enumerate(K_VIS, 1):
        for ci, (block, arm) in enumerate([(bp_block, "BP"), (rp_block, "RP")], 1):
            raw_flux = block.flux[STAR_IDX]
            wl = block.wavelengths

            fig.add_trace(go.Scatter(
                x=wl, y=raw_flux, mode="lines",
                line=dict(color="black", width=2, dash="dash"),
                name="Raw", legendgroup="raw",
                showlegend="raw" not in shown,
            ), row=ri, col=ci)
            shown.add("raw")

            for sigma in ALL_SIGMAS:
                smoothing = "none" if sigma == 0 else "gaussian"
                sm_kwargs = {} if sigma == 0 else {"sigma": sigma}
                smoothed = step02.smooth_flux(raw_flux, smoothing, **sm_kwargs)
                coeffs = step02.fit_basis(wl, smoothed, basis, K)
                recon = step02.reconstruct_flux(wl, coeffs, basis)

                lname = f"σ={sigma}"
                fig.add_trace(go.Scatter(
                    x=wl, y=recon, mode="lines",
                    line=dict(color=SIGMA_COLORS[sigma], width=1.6),
                    name=lname, legendgroup=lname,
                    showlegend=lname not in shown,
                ), row=ri, col=ci)
                shown.add(lname)

    fig.update_layout(
        height=700, width=1050,
        title_text=f"{basis.upper()} reconstruction — source_id={sid} (y={label})",
        template="plotly_white",
        legend=dict(orientation="h", yanchor="bottom", y=1.05, xanchor="center", x=0.5),
    )
    fig.update_xaxes(title_text="Wavelength (nm)", row=len(K_VIS))
    fig.update_yaxes(title_text="Flux", col=1)
    return fig

plot_basis_reconstructions("bspline").show()

### 10b. Chebyshev basis

In [56]:
plot_basis_reconstructions("chebyshev").show()

### 10c. Legendre basis

In [57]:
plot_basis_reconstructions("legendre").show()

### 10d. Smoothing only (before basis fitting)

What does Gaussian smoothing alone do to the raw spectrum? No basis fitting here — just raw vs smoothed at every σ.

In [58]:
fig_smooth = make_subplots(
    rows=1, cols=2,
    subplot_titles=["BP", "RP"],
    horizontal_spacing=0.06,
)
shown = set()

for ci, (block, arm) in enumerate([(bp_block, "BP"), (rp_block, "RP")], 1):
    raw_flux = block.flux[STAR_IDX]
    wl = block.wavelengths

    fig_smooth.add_trace(go.Scatter(
        x=wl, y=raw_flux, mode="lines",
        line=dict(color="black", width=2.5, dash="dash"),
        name="Raw", legendgroup="raw",
        showlegend="raw" not in shown,
    ), row=1, col=ci)
    shown.add("raw")

    for sigma in ALL_SIGMAS:
        if sigma == 0:
            continue
        smoothed = step02.smooth_flux(raw_flux, "gaussian", sigma=sigma)
        lname = f"σ={sigma}"
        fig_smooth.add_trace(go.Scatter(
            x=wl, y=smoothed, mode="lines",
            line=dict(color=SIGMA_COLORS[sigma], width=1.5),
            name=lname, legendgroup=lname,
            showlegend=lname not in shown,
        ), row=1, col=ci)
        shown.add(lname)

fig_smooth.update_layout(
    height=400, width=1050,
    title_text=f"Gaussian smoothing only — source_id={sid} (y={label})",
    template="plotly_white",
    legend=dict(orientation="h", yanchor="bottom", y=1.08, xanchor="center", x=0.5),
)
fig_smooth.update_xaxes(title_text="Wavelength (nm)")
fig_smooth.update_yaxes(title_text="Flux", col=1)
fig_smooth.show()